In [1]:
import os

In [2]:
%pwd

'/home/tuhin/Kedney-Disease-Classification/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/home/tuhin/Kedney-Disease-Classification'

In [5]:
os.environ["MLFLOW_TRACKING_URI"]="https://dagshub.com/tuhin1522/Kedney-Disease-Classification.mlflow/"
os.environ["MLFLOW_TRACKING_USERNAME"]="tuhin1522"
os.environ["MLFLOW_TRACKING_PASSWORD"]="22b91e97101996f521a3f0876d13a03b459b0a79"

In [6]:
import tensorflow as tf

I0000 00:00:1781381141.536499  105212 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1781381141.592643  105212 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781381143.038347  105212 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [7]:
model = tf.keras.models.load_model("artifacts/training/model.h5")

E0000 00:00:1781381149.159578  105212 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [8]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [9]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories, save_json

In [10]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    
    def get_evaluation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model="artifacts/training/model.h5",
            training_data="artifacts/data_ingestion/CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone",
            mlflow_uri="https://dagshub.com/tuhin1522/Kedney-Disease-Classification.mlflow/",
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config




In [11]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse

/home/tuhin/Kedney-Disease-Classification/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    
    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )


    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)
    

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = model.evaluate(self.valid_generator)
        self.save_score()

    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)

    
    def log_into_mlflow(self):
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        
        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics(
                {"loss": self.score[0], "accuracy": self.score[1]}
            )
            # Model registry does not work with file store
            if tracking_url_type_store != "file":

                # Register the model
                # There are other ways to use the Model Registry, which depends on the use case,
                # please refer to the doc for more information:
                # https://mlflow.org/docs/latest/model-registry.html#api-workflow
                mlflow.keras.log_model(self.model, "model", registered_model_name="VGG16Model")
            else:
                mlflow.keras.log_model(self.model, "model")


In [13]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()

except Exception as e:
   raise e

Found 3732 images belonging to 4 classes.


I0000 00:00:1781381371.461243  105212 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
W0000 00:00:1781381372.022801  111638 cpu_allocator_impl.cc:82] Allocation of 205520896 exceeds 10% of free system memory.
W0000 00:00:1781381372.114107  111638 cpu_allocator_impl.cc:82] Allocation of 205520896 exceeds 10% of free system memory.
W0000 00:00:1781381372.442169  111638 cpu_allocator_impl.cc:82] Allocation of 102760448 exceeds 10% of free system memory.
W0000 00:00:1781381372.548299  111638 cpu_allocator_impl.cc:82] Allocation of 102760448 exceeds 10% of free system memory.


  1/234 ━━━━━━━━━━━━━━━━━━━━ 9:51 3s/step - accuracy: 0.2500 - loss: 3.2082

W0000 00:00:1781381373.988666  111632 cpu_allocator_impl.cc:82] Allocation of 205520896 exceeds 10% of free system memory.


234/234 ━━━━━━━━━━━━━━━━━━━━ 921s 4s/step - accuracy: 0.5997 - loss: 6.2118


2026/06/14 02:25:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/14 02:25:09 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.
Successfully registered model 'VGG16Model'.
2026/06/14 02:26:27 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: VGG16Model, version 1
Created version '1' of model 'VGG16Model'.


🏃 View run indecisive-sow-411 at: https://dagshub.com/tuhin1522/Kedney-Disease-Classification.mlflow/#/experiments/0/runs/df01deac03364b329586a944f02c9c61
🧪 View experiment at: https://dagshub.com/tuhin1522/Kedney-Disease-Classification.mlflow/#/experiments/0
